In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DIR =", RAW_DIR)
print("CLEANED_DIR =", CLEANED_DIR)
print("RAW exists?", RAW_DIR.exists())
print("CLEANED exists?", CLEANED_DIR.exists())

PROJECT_ROOT = /Users/apple/Desktop/UCL/skills/python3
RAW_DIR = /Users/apple/Desktop/UCL/skills/python3/data/raw
CLEANED_DIR = /Users/apple/Desktop/UCL/skills/python3/data/cleaned
RAW exists? True
CLEANED exists? True


In [2]:
!python -m pip install requests beautifulsoup4 pandas tqdm nltk

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl.metadata (40 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl (309 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 27.5 MB/s  0:00:00 eta 0:00:01
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [4]:
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path

In [5]:
PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

In [6]:
landform_pages = [
    "Desert", "Canyon", "Glacier", "Volcano", "Cave",
    "Coral_reef", "Wetland", "Island", "Tundra", "Forest"
]

climate_pages = [
    "Monsoon", "Drought", "Erosion", "Sea_ice", "Sediment",
    "Floodplain", "Permafrost", "Atmosphere", "Storm", "Geothermal_energy"
]

In [7]:
import requests

session = requests.Session()
session.headers.update({
    "User-Agent": "DfPI-DigitalSkills-Project/1.0 (student project; contact: your_email@example.com)"
})

def fetch_wikipedia_extract(title: str):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "titles": title,
        "explaintext": 1,
        "redirects": 1
    }

    r = session.get(url, params=params, timeout=30)
    print("Status:", r.status_code, "|", title)

    r.raise_for_status()
    data = r.json()

    pages = data["query"]["pages"]
    page_data = list(pages.values())[0]

    return {
        "title": page_data.get("title", title),
        "text": page_data.get("extract", "")
    }

In [8]:
def split_into_paragraphs(text: str, min_words=40, max_words=250):
    raw_paragraphs = text.split("\n")
    cleaned = []

    for para in raw_paragraphs:
        para = re.sub(r"\[\d+\]", "", para)  # 去掉引用编号
        para = re.sub(r"\s+", " ", para).strip()

        word_count = len(para.split())
        if word_count < min_words or word_count > max_words:
            continue

        # 去掉非常像标题的短行
        if para.isupper():
            continue

        cleaned.append(para)

    return cleaned

In [9]:
landform_rows = []

for page in tqdm(landform_pages):
    result = fetch_wikipedia_extract(page)
    paragraphs = split_into_paragraphs(result["text"])

    for i, para in enumerate(paragraphs):
        landform_rows.append({
            "dataset": "landforms",
            "source": "wikipedia",
            "page_title": result["title"],
            "text_id": f"{page}_{i}",
            "raw_text": para
        })

landform_df = pd.DataFrame(landform_rows)
landform_df.head(), len(landform_df)

 20%|██        | 2/10 [00:00<00:01,  4.41it/s]

Status: 200 | Desert
Status: 200 | Canyon


 30%|███       | 3/10 [00:00<00:02,  3.26it/s]

Status: 200 | Glacier


 40%|████      | 4/10 [00:01<00:01,  3.27it/s]

...

 90%|█████████ | 9/10 [00:02<00:00,  4.89it/s]

Status: 200 | Island
Status: 200 | Tundra


100%|██████████| 10/10 [00:02<00:00,  3.90it/s]

Status: 200 | Forest


(     dataset     source page_title   text_id  \
 0  landforms  wikipedia     Desert  Desert_0   
 1  landforms  wikipedia     Desert  Desert_1   
 2  landforms  wikipedia     Desert  Desert_2   
 3  landforms  wikipedia     Desert  Desert_3   
 4  landforms  wikipedia     Desert  Desert_4   
 
                                             raw_text  
 0  A desert is a landscape where little precipita...  
 1  Deserts are formed by weathering processes as ...  
 2  Plants and animals living in the desert need s...  
 3  People have struggled to live in deserts and t...  
 4  English desert and its Romance cognates (inclu...  ,
 532)

In [10]:
climate_rows = []

for page in tqdm(climate_pages):
    result = fetch_wikipedia_extract(page)
    paragraphs = split_into_paragraphs(result["text"])

    for i, para in enumerate(paragraphs):
        climate_rows.append({
            "dataset": "climate_processes",
            "source": "wikipedia",
            "page_title": result["title"],
            "text_id": f"{page}_{i}",
            "raw_text": para
        })

climate_df = pd.DataFrame(climate_rows)
climate_df.head(), len(climate_df)

 10%|█         | 1/10 [00:00<00:01,  7.08it/s]

Status: 200 | Monsoon


 20%|██        | 2/10 [00:00<00:01,  5.12it/s]

Status: 200 | Drought


 30%|███       | 3/10 [00:00<00:01,  4.84it/s]

...

 90%|█████████ | 9/10 [00:01<00:00,  4.54it/s]

Status: 200 | Storm


100%|██████████| 10/10 [00:02<00:00,  4.51it/s]

Status: 200 | Geothermal_energy


(             dataset     source page_title    text_id  \
 0  climate_processes  wikipedia    Monsoon  Monsoon_0   
 1  climate_processes  wikipedia    Monsoon  Monsoon_1   
 2  climate_processes  wikipedia    Monsoon  Monsoon_2   
 3  climate_processes  wikipedia    Monsoon  Monsoon_3   
 4  climate_processes  wikipedia    Monsoon  Monsoon_4   
 
                                             raw_text  
 0  A monsoon () is traditionally a seasonal rever...  
 1  Strengthening of the Asian monsoon has been li...  
 2  A study of marine plankton suggested that the ...  
 3  Five episodes during the Quaternary at 2.22 Ma...  
 4  Monsoons were once considered as a large-scale...  ,
 359)

In [11]:
print("Landforms:", len(landform_df))
print("Climate:", len(climate_df))

Landforms: 532
Climate: 359


In [12]:
landform_df.to_csv(RAW_DIR / "wikipedia_landforms.csv", index=False)
climate_df.to_csv(RAW_DIR / "wikipedia_climate.csv", index=False)

In [13]:
gutenberg_dir = RAW_DIR / "gutenberg_books"
book_paths = list(gutenberg_dir.glob("*.txt"))

print(book_paths)

[PosixPath('/Users/apple/Desktop/UCL/skills/python3/data/raw/gutenberg_books/canoeing_in_the_wilderness.txt'), PosixPath('/Users/apple/Desktop/UCL/skills/python3/data/raw/gutenberg_books/the_yosemite.txt'), PosixPath('/Users/apple/Desktop/UCL/skills/python3/data/raw/gutenberg_books/thousand_mile_walk.txt')]


In [14]:
def clean_gutenberg_text(text: str):
    start_patterns = [
        r"\*\*\* START OF (THE|THIS) PROJECT GUTENBERG EBOOK .* \*\*\*",
        r"\*\*\*START OF (THE|THIS) PROJECT GUTENBERG EBOOK .* \*\*\*"
    ]
    end_patterns = [
        r"\*\*\* END OF (THE|THIS) PROJECT GUTENBERG EBOOK .* \*\*\*",
        r"\*\*\*END OF (THE|THIS) PROJECT GUTENBERG EBOOK .* \*\*\*"
    ]

    for pattern in start_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            text = text[match.end():]
            break

    for pattern in end_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            text = text[:match.start()]
            break

    return text.strip()

In [15]:
def split_book_paragraphs(text: str, min_words=40, max_words=250):
    paragraphs = re.split(r"\n\s*\n", text)
    output = []

    for para in paragraphs:
        para = re.sub(r"\s+", " ", para).strip()
        wc = len(para.split())

        if wc < min_words or wc > max_words:
            continue

        output.append(para)

    return output

In [16]:
gutenberg_rows = []

for path in book_paths:
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = clean_gutenberg_text(text)
    paragraphs = split_book_paragraphs(text)

    for i, para in enumerate(paragraphs):
        gutenberg_rows.append({
            "dataset": "gutenberg_nature_writing",
            "source": "project_gutenberg",
            "page_title": path.stem,
            "text_id": f"{path.stem}_{i}",
            "raw_text": para
        })

gutenberg_df = pd.DataFrame(gutenberg_rows)
print("Gutenberg total rows:", len(gutenberg_df))
gutenberg_df.head()

Gutenberg total rows: 904


,dataset,source,page_title,text_id,raw_text
0,gutenberg_nature_writing,project_gutenberg,canoeing_in_the_wilderness,canoeing_in_the_wilderness_0,"Thoreau was born at Concord, Massachusetts, Ju..."
1,gutenberg_nature_writing,project_gutenberg,canoeing_in_the_wilderness,canoeing_in_the_wilderness_1,The chief attraction that inspired Thoreau to ...
2,gutenberg_nature_writing,project_gutenberg,canoeing_in_the_wilderness,canoeing_in_the_wilderness_2,He liked especially the companionship of men w...
3,gutenberg_nature_writing,project_gutenberg,canoeing_in_the_wilderness,canoeing_in_the_wilderness_3,Thoreau was one of the world’s greatest nature...
4,gutenberg_nature_writing,project_gutenberg,canoeing_in_the_wilderness,canoeing_in_the_wilderness_4,It is to be noted that he was no hunter. His i...


In [17]:
gutenberg_df.to_csv(RAW_DIR / "gutenberg_nature.csv", index=False)
print("已保存到:", RAW_DIR / "gutenberg_nature.csv")

已保存到: /Users/apple/Desktop/UCL/skills/python3/data/raw/gutenberg_nature.csv
